# jupyter_lab_extractor — Test Notebook

This notebook tests the `%%extract` cell magic from the `jupyter_lab_extractor` package.

**Note:** This notebook is paired with a `.py` script via Jupytext.
Always run from the `.ipynb` — the `.py` file is for version control / diffing only.
Running the `.py` directly will not work since cell magics require a live Jupyter kernel.

## What this notebook covers
1. Writing cell contents to a new file
2. Appending to an existing file with `-a`
3. Overwriting an existing file (default `-w` behavior)
4. Using `%%extract` and `%%ipytest` together on the same cell
5. Metadata headers and magic line stripping
6. Error handling

---
# Setup

## Logging with Loguru
Provides visibility into what is happening during test execution.

In [1]:
# Logging configuration
# Log levels from most output to least (severity low to high):
# TRACE, DEBUG, INFO, SUCCESS, WARNING, ERROR, CRITICAL
CONSOLE_LOG_LEVEL = "DEBUG"
FILE_LOG_LEVEL = "DEBUG"

from loguru import logger
from pathlib import Path
import sys

# Remove default handler to avoid duplicates
logger.remove()

# Configure log file
LOG_FILE = Path.cwd() / "test_extract_debug.log"

# Clear existing log file to start fresh each run
if LOG_FILE.exists():
    LOG_FILE.unlink()

# Console handler configuration - colorful output for Jupyter
console_handler_id = logger.add(
    sys.stdout,
    format="<level>{level: <8}</level> | <level>{message}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan> | <green>{time:HH:mm:ss.SSS}</green>",
    level=CONSOLE_LOG_LEVEL,
    colorize=True,
    enqueue=False,  # Must be False for Jupyter compatibility
    backtrace=True,
    diagnose=True
)

# File handler configuration - single file, overwritten each run
file_handler_id = logger.add(
    LOG_FILE,
    format="{time:YYYY-MM-DD HH:mm:ss.SSS} | {level: <8} | {process.id}:{thread.id} | {name}:{function}:{line} | {message}",
    level=FILE_LOG_LEVEL,
    enqueue=True,  # Thread-safe for file operations
    backtrace=True,
    diagnose=True
)

logger.success("Loguru configured successfully")
logger.info(f"Log file: {LOG_FILE.absolute()}")
logger.info(f"Console handler ID: {console_handler_id}, File handler ID: {file_handler_id}")

SUCCESS  | Loguru configured successfully | __main__:<module> | 21:14:38.817
INFO     | Log file: /media/StorageLW/GShared/Project_restart/jupyter_lab_extractor/tests/test_extract_debug.log | __main__:<module> | 21:14:38.818
INFO     | Console handler ID: 1, File handler ID: 2 | __main__:<module> | 21:14:38.819


## Imports and ipytest Configuration

In [2]:
import os
import shutil
import ipytest
ipytest.autoconfig()

logger.success("ipytest configured")

SUCCESS  | ipytest configured | __main__:<module> | 21:14:38.859


## Load the `%%extract` Magic

In [3]:
%load_ext jupyter_lab_extractor
logger.success("jupyter_lab_extractor magic loaded")

SUCCESS  | jupyter_lab_extractor magic loaded | __main__:<module> | 21:14:38.883


## Prepare Output Directory
All extracted files go into a subfolder to keep the test directory clean.

In [4]:
OUTPUT_DIR = Path("test_demo_outputs")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
    logger.debug(f"Cleared existing {OUTPUT_DIR}/")
OUTPUT_DIR.mkdir(exist_ok=True)
logger.success(f"Output directory ready: {OUTPUT_DIR}/")

SUCCESS  | Output directory ready: test_demo_outputs/ | __main__:<module> | 21:14:38.886


# Usage Examples (with quick confirm test)
These cells use `%%extract` as a user would in a real notebook,
then verify the output with ipytest.

## Test 1: Write Cell Contents to a New File
The default behavior (`-w`) should create a new file with the cell contents
and a metadata header comment.

In [5]:
%%extract test_demo_outputs/test_output_1.py
x = 42
y = "hello"

In [6]:
logger.info("Wrote test_demo_outputs/test_output_1.py — checking contents:")
logger.debug(open("test_demo_outputs/test_output_1.py").read())

INFO     | Wrote test_demo_outputs/test_output_1.py — checking contents: | __main__:<module> | 21:14:38.892
DEBUG    | # Source: tests/test_extract_magic.ipynb | Cell In[5] | 2026-02-01 21:14:38
x = 42
y = "hello"

 | __main__:<module> | 21:14:38.893


In [7]:
%%ipytest

from loguru import logger

def test_write_new_file():
    content = open("test_demo_outputs/test_output_1.py").read()
    assert "x = 42" in content
    logger.debug("Found 'x = 42'")
    assert 'y = "hello"' in content
    logger.debug("Found 'y = \"hello\"'")
    assert "# Source:" in content
    logger.debug("Found metadata header")
    logger.success("test_write_new_file passed")

DEBUG    | Found 'x = 42' | __main__:test_write_new_file | 21:14:39.087
DEBUG    | Found 'y = "hello"' | __main__:test_write_new_file | 21:14:39.087
DEBUG    | Found metadata header | __main__:test_write_new_file | 21:14:39.088
SUCCESS  | test_write_new_file passed | __main__:test_write_new_file | 21:14:39.089
.                                                                                            [100%]
1 passed in 0.01s


## Test 2: Write Then Append
First cell creates `test_output_2.py`, second cell appends to it with `-a`.
The result should contain both blocks with two metadata headers.

### Write the initial file

In [8]:
%%extract test_demo_outputs/test_output_2.py
import os
CONSTANT = 100

In [9]:
logger.info("Wrote test_demo_outputs/test_output_2.py — initial block")

INFO     | Wrote test_demo_outputs/test_output_2.py — initial block | __main__:<module> | 21:14:39.183


### Append a second block

In [10]:
%%extract test_demo_outputs/test_output_2.py -a
def helper():
    return CONSTANT * 2

In [11]:
logger.info("Appended to test_demo_outputs/test_output_2.py — checking contents:")
logger.debug(open("test_demo_outputs/test_output_2.py").read())

INFO     | Appended to test_demo_outputs/test_output_2.py — checking contents: | __main__:<module> | 21:14:39.188
DEBUG    | # Source: tests/test_extract_magic.ipynb | Cell In[8] | 2026-02-01 21:14:39
import os
CONSTANT = 100

# Source: tests/test_extract_magic.ipynb | Cell In[10] | 2026-02-01 21:14:39
def helper():
    return CONSTANT * 2

 | __main__:<module> | 21:14:39.189


### Confirm both blocks are present

In [12]:
%%ipytest

from loguru import logger

def test_write_then_append():
    content = open("test_demo_outputs/test_output_2.py").read()
    assert "import os" in content
    logger.debug("Found 'import os'")
    assert "CONSTANT = 100" in content
    logger.debug("Found 'CONSTANT = 100'")
    assert "def helper():" in content
    logger.debug("Found 'def helper():'")
    assert "return CONSTANT * 2" in content
    logger.debug("Found 'return CONSTANT * 2'")
    assert content.count("# Source:") == 2
    logger.debug("Found 2 metadata headers")
    logger.success("test_write_then_append passed")

DEBUG    | Found 'import os' | __main__:test_write_then_append | 21:14:39.237
DEBUG    | Found 'CONSTANT = 100' | __main__:test_write_then_append | 21:14:39.237
DEBUG    | Found 'def helper():' | __main__:test_write_then_append | 21:14:39.238
DEBUG    | Found 'return CONSTANT * 2' | __main__:test_write_then_append | 21:14:39.238
DEBUG    | Found 2 metadata headers | __main__:test_write_then_append | 21:14:39.238
SUCCESS  | test_write_then_append passed | __main__:test_write_then_append | 21:14:39.239
.                                                                                            [100%]
1 passed in 0.01s


## Test 3: Overwrite Replaces Existing Content
Copy `test_output_2.py` (which has two blocks), then overwrite the copy.
The old content should be completely gone.

### Make a copy to work with

In [13]:
shutil.copy("test_demo_outputs/test_output_2.py", "test_demo_outputs/test_output_2_copy.py")
logger.info("Copied test_output_2.py -> test_output_2_copy.py")

INFO     | Copied test_output_2.py -> test_output_2_copy.py | __main__:<module> | 21:14:39.334


### Overwrite the copy with new content

In [14]:
%%extract test_demo_outputs/test_output_2_copy.py
completely_new = True

In [15]:
logger.info("Overwrote test_demo_outputs/test_output_2_copy.py — checking contents:")
logger.debug(open("test_demo_outputs/test_output_2_copy.py").read())

INFO     | Overwrote test_demo_outputs/test_output_2_copy.py — checking contents: | __main__:<module> | 21:14:39.340
DEBUG    | # Source: tests/test_extract_magic.ipynb | Cell In[14] | 2026-02-01 21:14:39
completely_new = True

 | __main__:<module> | 21:14:39.341


### Confirm old content is gone

In [16]:
%%ipytest

from loguru import logger

def test_overwrite_copy():
    content = open("test_demo_outputs/test_output_2_copy.py").read()
    assert "completely_new = True" in content
    logger.debug("Found 'completely_new = True'")
    assert "CONSTANT" not in content
    logger.debug("Confirmed old 'CONSTANT' is gone")
    assert "def helper" not in content
    logger.debug("Confirmed old 'def helper' is gone")
    assert content.count("# Source:") == 1
    logger.debug("Found exactly 1 metadata header")
    logger.success("test_overwrite_copy passed")

DEBUG    | Found 'completely_new = True' | __main__:test_overwrite_copy | 21:14:39.389
DEBUG    | Confirmed old 'CONSTANT' is gone | __main__:test_overwrite_copy | 21:14:39.390
DEBUG    | Confirmed old 'def helper' is gone | __main__:test_overwrite_copy | 21:14:39.390
DEBUG    | Found exactly 1 metadata header | __main__:test_overwrite_copy | 21:14:39.390
SUCCESS  | test_overwrite_copy passed | __main__:test_overwrite_copy | 21:14:39.391
.                                                                                            [100%]
1 passed in 0.01s


## Test 4: Extract + ipytest Combo
This cell is both extracted to a file AND run as a test simultaneously.
Demonstrates that `%%extract` does not interfere with cell execution,
even when another cell magic (`%%ipytest`) is present in the cell body.

In [17]:
%%extract test_demo_outputs/extracted_test.py
%%ipytest

from loguru import logger

def test_round_trip():
    """This test was both run by ipytest AND extracted to a file"""
    assert 1 + 1 == 2
    logger.debug("1 + 1 == 2")
    assert "hello".upper() == "HELLO"
    logger.debug("'hello'.upper() == 'HELLO'")
    logger.success("test_round_trip passed — cell was extracted and executed")

DEBUG    | 1 + 1 == 2 | __main__:test_round_trip | 21:14:39.531
DEBUG    | 'hello'.upper() == 'HELLO' | __main__:test_round_trip | 21:14:39.532
SUCCESS  | test_round_trip passed — cell was extracted and executed | __main__:test_round_trip | 21:14:39.532
.                                                                                            [100%]
1 passed in 0.01s


In [18]:
logger.info("Checking extracted_test.py contents:")
logger.debug(open("test_demo_outputs/extracted_test.py").read())

INFO     | Checking extracted_test.py contents: | __main__:<module> | 21:14:39.627
DEBUG    | # Source: tests/test_extract_magic.ipynb | Cell In[17] | 2026-02-01 21:14:39

from loguru import logger

def test_round_trip():
    """This test was both run by ipytest AND extracted to a file"""
    assert 1 + 1 == 2
    logger.debug("1 + 1 == 2")
    assert "hello".upper() == "HELLO"
    logger.debug("'hello'.upper() == 'HELLO'")
    logger.success("test_round_trip passed — cell was extracted and executed")

 | __main__:<module> | 21:14:39.628


---
# Deeper Unit Tests
These use `tmp_path` fixtures and `run_cell_magic()` directly
for more isolated testing.

## Overwrite Mode (default)

In [19]:
%%ipytest

import os
from loguru import logger

def test_extract_overwrite(tmp_path):
    """Test that default mode overwrites the file"""
    target = str(tmp_path / "out.py")
    ip = get_ipython()

    # Write something first
    with open(target, 'w') as f:
        f.write("old content\n")
    logger.debug(f"Wrote 'old content' to {target}")

    ip.run_cell_magic('extract', target, 'x = 1')
    logger.debug(f"Ran %%extract on {target}")

    content = open(target).read()
    assert "old content" not in content
    logger.debug("Confirmed 'old content' was overwritten")
    assert "x = 1" in content
    logger.debug("Found 'x = 1'")
    assert "# Source:" in content
    logger.debug("Found metadata header")
    logger.success("test_extract_overwrite passed")

DEBUG    | Wrote 'old content' to /tmp/pytest-of-ukde/pytest-20/test_extract_overwrite0/out.py | __main__:test_extract_overwrite | 21:14:39.680
DEBUG    | Ran %%extract on /tmp/pytest-of-ukde/pytest-20/test_extract_overwrite0/out.py | __main__:test_extract_overwrite | 21:14:39.682
DEBUG    | Confirmed 'old content' was overwritten | __main__:test_extract_overwrite | 21:14:39.682
DEBUG    | Found 'x = 1' | __main__:test_extract_overwrite | 21:14:39.683
DEBUG    | Found metadata header | __main__:test_extract_overwrite | 21:14:39.683
SUCCESS  | test_extract_overwrite passed | __main__:test_extract_overwrite | 21:14:39.684
.                                                                                            [100%]
1 passed in 0.01s


## Append Mode (`-a`)

In [20]:
%%ipytest

from loguru import logger

def test_extract_append(tmp_path):
    """Test that -a appends to the file"""
    target = str(tmp_path / "out.py")

    ip = get_ipython()
    ip.run_cell_magic('extract', target, 'x = 1')
    logger.debug(f"Wrote first block to {target}")
    ip.run_cell_magic('extract', f'{target} -a', 'y = 2')
    logger.debug(f"Appended second block to {target}")

    content = open(target).read()
    assert "x = 1" in content
    assert "y = 2" in content
    assert content.count("# Source:") == 2
    logger.debug("Found both blocks and 2 metadata headers")
    logger.success("test_extract_append passed")

DEBUG    | Wrote first block to /tmp/pytest-of-ukde/pytest-21/test_extract_append0/out.py | __main__:test_extract_append | 21:14:39.830
DEBUG    | Appended second block to /tmp/pytest-of-ukde/pytest-21/test_extract_append0/out.py | __main__:test_extract_append | 21:14:39.831
DEBUG    | Found both blocks and 2 metadata headers | __main__:test_extract_append | 21:14:39.831
SUCCESS  | test_extract_append passed | __main__:test_extract_append | 21:14:39.831
.                                                                                            [100%]
1 passed in 0.01s


## Magic Lines Are Stripped

In [21]:
%%ipytest

from loguru import logger

def test_magic_lines_stripped(tmp_path):
    """Test that % and %% magic lines are removed from output"""
    target = str(tmp_path / "out.py")

    ip = get_ipython()
    cell_content = "%matplotlib inline\nimport numpy as np\n%%time\nx = 1"
    ip.run_cell_magic('extract', target, cell_content)
    logger.debug(f"Extracted cell with mixed magic lines to {target}")

    content = open(target).read()
    assert "matplotlib" not in content
    logger.debug("Confirmed '%matplotlib inline' was stripped")
    assert "%%time" not in content
    logger.debug("Confirmed '%%time' was stripped")
    assert "import numpy as np" in content
    logger.debug("Confirmed 'import numpy as np' was kept")
    assert "x = 1" in content
    logger.debug("Confirmed 'x = 1' was kept")
    logger.success("test_magic_lines_stripped passed")

DEBUG    | Extracted cell with mixed magic lines to /tmp/pytest-of-ukde/pytest-22/test_magic_lines_stripped0/out.py | __main__:test_magic_lines_stripped | 21:14:40.214
DEBUG    | Confirmed '%matplotlib inline' was stripped | __main__:test_magic_lines_stripped | 21:14:40.215
DEBUG    | Confirmed '%%time' was stripped | __main__:test_magic_lines_stripped | 21:14:40.216
DEBUG    | Confirmed 'import numpy as np' was kept | __main__:test_magic_lines_stripped | 21:14:40.216
DEBUG    | Confirmed 'x = 1' was kept | __main__:test_magic_lines_stripped | 21:14:40.216
SUCCESS  | test_magic_lines_stripped passed | __main__:test_magic_lines_stripped | 21:14:40.217
.                                                                                            [100%]
1 passed in 0.25s


## Missing Filename Raises Error

In [22]:
%%ipytest

import pytest
from loguru import logger

def test_extract_no_filename():
    """Test that missing filename raises ValueError"""
    ip = get_ipython()
    with pytest.raises(ValueError):
        ip.run_cell_magic('extract', '', 'x = 1')
    logger.success("test_extract_no_filename passed — ValueError raised as expected")

SUCCESS  | test_extract_no_filename passed — ValueError raised as expected | __main__:test_extract_no_filename | 21:14:40.437
.                                                                                            [100%]
1 passed in 0.01s


## Metadata Header Format

In [23]:
%%ipytest

from loguru import logger

def test_metadata_header(tmp_path):
    """Test that header contains expected metadata fields"""
    target = str(tmp_path / "out.py")

    ip = get_ipython()
    ip.run_cell_magic('extract', target, 'x = 1')

    content = open(target).read()
    header = content.splitlines()[0]
    logger.debug(f"Header: {header}")
    assert header.startswith("# Source:")
    logger.debug("Header starts with '# Source:'")
    assert "Cell In[" in header
    logger.debug("Header contains cell execution number")
    assert "|" in header
    logger.debug("Header contains pipe delimiters")
    logger.success("test_metadata_header passed")

DEBUG    | Header: # Source: tests/test_extract_magic.ipynb | Cell In[23] | 2026-02-01 21:14:40 | __main__:test_metadata_header | 21:14:40.637
DEBUG    | Header starts with '# Source:' | __main__:test_metadata_header | 21:14:40.638
DEBUG    | Header contains cell execution number | __main__:test_metadata_header | 21:14:40.638
DEBUG    | Header contains pipe delimiters | __main__:test_metadata_header | 21:14:40.638
SUCCESS  | test_metadata_header passed | __main__:test_metadata_header | 21:14:40.639
.                                                                                            [100%]
1 passed in 0.01s
